# Tokenization Benchmark — BM25

Compares six analyzers (Whitespace, Regex, NLTK Treebank, spaCy, Regex + Porter,
Regex + WordNet lemma) with an independent BM25 index per analyzer, on every
dataset listed in `DATASETS` below.

| Dataset | Corpus | Evaluated queries | Relevance |
|---|---|---|---|
| `quora` (BEIR) | 522,931 questions | 5,000 dev | binary |
| `hotpotqa` (BEIR) | 5,233,329 Wikipedia paragraphs | 5,447 dev | binary, 2 docs/query |
| `wands` (Wayfair) | 42,994 products | 480 | graded (Exact=2, Partial=1) |

Each dataset section is skipped unless its name is in `DATASETS`.

In [ ]:
# Datasets to benchmark, in run order. Remove any you want to skip.
# Available: "quora", "hotpotqa", "wands"
DATASETS = ["quora", "hotpotqa", "wands"]

K = 100          # retrieval depth
BATCH_SIZE = 64  # queries per retrieve() call

BM25_PARAMS = dict(method="lucene", k1=1.2, b=0.75)

AVAILABLE_DATASETS = {"quora", "hotpotqa", "wands"}
unknown = set(DATASETS) - AVAILABLE_DATASETS
assert not unknown, f"Unknown datasets: {unknown}"

In [ ]:
import gc
import re
import time
from collections import defaultdict
from functools import partial
from pathlib import Path

import bm25s
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import spacy
from datasets import load_dataset
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TreebankWordTokenizer
from tqdm.auto import tqdm

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

INDEX_ROOT = Path("indices")

# Filled by each dataset section: {dataset_name: results_df}
all_dataset_results = {}

## Tokenizers

In [ ]:
treebank = TreebankWordTokenizer()
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()
nlp = spacy.blank("en")


# 1. Whitespace
def whitespace_tokenizer(text):
    return text.lower().split()


# 2. Regex
def regex_tokenizer(text):
    return re.findall(r"\w+", text.lower())


# 3. NLTK Treebank
def nltk_tokenizer(text):
    return treebank.tokenize(text.lower())


# 4. spaCy
def spacy_tokenizer(text):
    return [
        token.text.lower()
        for token in nlp.make_doc(text)
        if not token.is_space
    ]


# 5. Regex + Porter stemming
def porter_tokenizer(text):
    return [porter.stem(token) for token in regex_tokenizer(text)]


# 6. Regex + WordNet lemmatization
def lemma_tokenizer(text):
    return [lemmatizer.lemmatize(token) for token in regex_tokenizer(text)]


tokenizers = {
    "Whitespace": whitespace_tokenizer,
    "Regex": regex_tokenizer,
    "NLTK": nltk_tokenizer,
    "spaCy": spacy_tokenizer,
    "Regex_Porter": porter_tokenizer,
    "Regex_Lemma": lemma_tokenizer,
}

### Tokenizer Examples

In [ ]:
examples = [
    # Quora-style questions
    "How can I learn Python programming?",
    "What's the best way to learn machine-learning?",
    "Is Python 3.11 better than Python 2.7?",
    "C++ vs. C#: Which language should I learn?",
    # WANDS-style product queries
    "60-inch Stainless Steel Tables",
    "women's running shoes",
    "Dining table, 4 chairs (set of 5)",
    # Stemming vs. lemmatization
    "How are programmers writing better programs?",
    "Stainless steel tables with storage drawers",
]

for example in examples:
    print("\nINPUT:", example)

    for name, tokenizer in tokenizers.items():
        print(f"{name:13}", tokenizer(example))

## Evaluation Metrics

- **Binary** (Quora, HotpotQA): nDCG@10, MRR@10, Recall@100. HotpotQA also reports
  **Both@10**: both supporting paragraphs appear in the top 10.
- **Graded** (WANDS): nDCG@10 with gain `2^rel - 1`, MRR@10 over Exact matches only,
  Recall@10 and Recall@100 over Exact + Partial.

Queries where a metric is undefined return `NaN` for it and are skipped by the mean.

In [ ]:
def evaluate_binary(retrieved_ids, query_qrels, both_at_10=False):
    relevant = {
        str(doc_id)
        for doc_id, score in query_qrels.items()
        if score > 0
    }

    if not relevant:
        return None

    top_10 = retrieved_ids[:10]

    recall_100 = len(set(retrieved_ids[:100]) & relevant) / len(relevant)

    mrr_10 = 0.0
    for rank, doc_id in enumerate(top_10, start=1):
        if doc_id in relevant:
            mrr_10 = 1.0 / rank
            break

    rels = np.array([int(doc_id in relevant) for doc_id in top_10])
    dcg = np.sum(rels / np.log2(np.arange(2, len(rels) + 2)))

    ideal_count = min(len(relevant), 10)
    idcg = np.sum(1 / np.log2(np.arange(2, ideal_count + 2)))

    metrics = {
        "nDCG@10": dcg / idcg if idcg > 0 else 0.0,
        "MRR@10": mrr_10,
        "Recall@100": recall_100,
    }

    if both_at_10:
        metrics["Both@10"] = float(relevant.issubset(set(top_10)))

    return metrics


def dcg_at_k(relevances, k):
    rel = np.asarray(relevances[:k], dtype=float)

    if len(rel) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(rel) + 2))

    return float(np.sum((2 ** rel - 1) / discounts))


def evaluate_graded(retrieved_ids, query_qrels):
    # Known relevant documents (Exact or Partial)
    relevant = {pid for pid, rel in query_qrels.items() if rel > 0}
    exact = {pid for pid, rel in query_qrels.items() if rel == 2}

    recall_10 = (
        len(set(retrieved_ids[:10]) & relevant) / len(relevant)
        if relevant else np.nan
    )

    recall_100 = (
        len(set(retrieved_ids[:100]) & relevant) / len(relevant)
        if relevant else np.nan
    )

    # MRR@10 (Exact-only)
    mrr_10 = np.nan
    if exact:
        mrr_10 = 0.0
        for rank, pid in enumerate(retrieved_ids[:10], start=1):
            if pid in exact:
                mrr_10 = 1.0 / rank
                break

    ranked_rels = [query_qrels.get(pid, 0) for pid in retrieved_ids[:10]]
    ideal_dcg = dcg_at_k(sorted(query_qrels.values(), reverse=True), 10)

    ndcg_10 = (
        dcg_at_k(ranked_rels, 10) / ideal_dcg
        if ideal_dcg > 0 else np.nan
    )

    return {
        "nDCG@10": ndcg_10,
        "MRR@10": mrr_10,
        "Recall@10": recall_10,
        "Recall@100": recall_100,
    }

## Benchmark Runner

One function shared by every dataset: tokenize corpus → build and save a BM25 index →
tokenize queries → batched retrieval → per-query metrics → aggregate. Results are
checkpointed to `<dataset>_tokenizer_results.csv` and `<dataset>_per_query_results.csv`
after each analyzer, and indices are saved under `indices/<dataset>/<analyzer>`.

In [ ]:
def run_benchmark(
    dataset_name,
    corpus_texts,
    n_docs,
    doc_ids,
    query_ids,
    query_texts,
    qrels,
    evaluate_fn,
):
    """
    corpus_texts: zero-argument callable returning an iterable of document
    texts in the same order as doc_ids. A callable lets large corpora
    stream from the HF dataset instead of being held as one list of strings.
    """
    index_dir = INDEX_ROOT / dataset_name
    index_dir.mkdir(parents=True, exist_ok=True)

    results_path = f"{dataset_name}_tokenizer_results.csv"
    per_query_path = f"{dataset_name}_per_query_results.csv"

    all_results = []
    per_query_results = []

    for name, tokenizer in tokenizers.items():

        print(f"\n{'=' * 60}")
        print(f"{dataset_name.upper()} | ANALYZER: {name}")
        print(f"{'=' * 60}")

        # 1. Tokenize corpus
        start = time.perf_counter()

        corpus_tokens = [
            tokenizer(text)
            for text in tqdm(
                corpus_texts(),
                total=n_docs,
                desc=f"Tokenizing {name}"
            )
        ]

        tokenization_time = time.perf_counter() - start

        # 2. Build an independent BM25 index
        start = time.perf_counter()

        retriever = bm25s.BM25(**BM25_PARAMS)
        retriever.index(corpus_tokens)

        indexing_time = time.perf_counter() - start

        # 3. Save index
        retriever.save(str(index_dir / name))

        del corpus_tokens
        gc.collect()

        # 4. Tokenize queries
        start = time.perf_counter()

        query_tokens = [tokenizer(query) for query in query_texts]

        query_tokenization_time = time.perf_counter() - start

        # 5. Batched retrieval
        query_metrics = []
        retrieval_start = time.perf_counter()

        for batch_start in tqdm(
            range(0, len(query_tokens), BATCH_SIZE),
            desc=f"Retrieving {name}"
        ):
            batch_end = batch_start + BATCH_SIZE

            # Numerical corpus positions
            doc_indices, scores = retriever.retrieve(
                query_tokens[batch_start:batch_end],
                k=K,
                show_progress=False
            )

            for local_idx, ranked_indices in enumerate(doc_indices):
                query_idx = batch_start + local_idx
                qid = query_ids[query_idx]

                retrieved_ids = [doc_ids[int(i)] for i in ranked_indices]

                metrics = evaluate_fn(retrieved_ids, qrels.get(qid, {}))

                if metrics is None:
                    continue

                query_metrics.append(metrics)

                per_query_results.append({
                    "Tokenizer": name,
                    "query_id": qid,
                    "query": query_texts[query_idx],
                    **metrics
                })

        retrieval_time = time.perf_counter() - retrieval_start

        # 6. Aggregate (NaN metrics are skipped by the mean)
        metric_df = pd.DataFrame(query_metrics)

        if metric_df.empty:
            raise ValueError("No evaluated queries. Check query/qrels IDs.")

        result = {
            "Tokenizer": name,
            **metric_df.mean().to_dict(),
            "TokenizationTime(s)": tokenization_time,
            "IndexTime(s)": indexing_time,
            "AvgQueryTime(ms)": retrieval_time / len(query_ids) * 1000,
            "QueryTokenizationTime(s)": query_tokenization_time,
            "EvaluatedQueries": int(metric_df["nDCG@10"].notna().sum()),
        }

        all_results.append(result)
        print(pd.Series(result))

        # Checkpoint results after each analyzer
        pd.DataFrame(all_results).to_csv(results_path, index=False)
        pd.DataFrame(per_query_results).to_csv(per_query_path, index=False)

        del retriever, query_tokens
        gc.collect()

    results_df = (
        pd.DataFrame(all_results)
        .sort_values("nDCG@10", ascending=False)
        .reset_index(drop=True)
    )

    results_df.to_csv(results_path, index=False)
    per_query_df = pd.DataFrame(per_query_results)

    display(results_df)

    return results_df, per_query_df

### Visualization Helpers

In [ ]:
# Palette: one categorical slot per metric, fixed order, light surface
SURFACE, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

METRIC_COLORS = {
    "nDCG@10": "#2a78d6",
    "MRR@10": "#eb6834",
    "Recall@100": "#1baf7a",
    "Recall@10": "#eda100",
    "Both@10": "#8a5cd1",
}


def result_metrics(results_df):
    return [m for m in METRIC_COLORS if m in results_df.columns]


def plot_results(results_df, title):
    metric_cols = result_metrics(results_df)

    # Best tokenizer on top
    plot_df = results_df.set_index("Tokenizer").iloc[::-1]

    ncols = 2
    nrows = int(np.ceil(len(metric_cols) / ncols))

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(13, 4 * nrows),
        sharey=True,
        facecolor=SURFACE,
        squeeze=False
    )
    axes = axes.flatten()

    for ax, metric in zip(axes, metric_cols):
        color = METRIC_COLORS[metric]
        ax.set_facecolor(SURFACE)

        ax.barh(plot_df.index, plot_df[metric], height=0.62, color=color)

        # Panels have different scales (e.g. Recall@10 is small by construction)
        top = plot_df[metric].max()
        ax.set_xlim(0, top * 1.22)

        for y, value in enumerate(plot_df[metric]):
            ax.text(
                value + top * 0.015, y, f"{value:.3f}",
                va="center", fontsize=9.5, color=INK2
            )

        ax.set_title(metric, loc="left", fontsize=12, fontweight="bold", color=INK)

        ax.set_axisbelow(True)
        ax.xaxis.grid(True, color=GRID, linewidth=0.8)
        ax.yaxis.grid(False)
        ax.tick_params(axis="x", colors=MUTED)
        ax.tick_params(axis="y", colors=INK2, length=0)

        for side in ("top", "right", "left"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color("#c3c2b7")

    # Hide unused panels
    for ax in axes[len(metric_cols):]:
        ax.set_visible(False)

    fig.suptitle(title, x=0.01, ha="left", fontsize=14, fontweight="bold", color=INK)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()


def print_improvement(results_df, baseline_name="Whitespace"):
    best = results_df.iloc[0]
    baseline = results_df.loc[results_df["Tokenizer"] == baseline_name].iloc[0]

    print("Best tokenizer:", best["Tokenizer"])
    print(f"nDCG@10: {best['nDCG@10']:.4f}")
    print(f"Recall@100: {best['Recall@100']:.4f}")

    print(f"\nImprovement vs {baseline_name} (percentage points):")

    for metric in result_metrics(results_df):
        print(f"{metric}: {(best[metric] - baseline[metric]) * 100:+.2f} pp")

---
## Dataset 1: BEIR Quora

522,931 questions, 5,000 dev queries, binary relevance (duplicate questions).
Document text = `title + " " + text`.

### Load Data

In [ ]:
if "quora" in DATASETS:
    corpus_ds = load_dataset("BeIR/quora", "corpus", split="corpus")
    queries_ds = load_dataset("BeIR/quora", "queries", split="queries")
    qrels_dev_ds = load_dataset("BeIR/quora-qrels", split="validation")

    # Preserve document order
    doc_ids = [str(x) for x in corpus_ds["_id"]]

    doc_texts = [
        f"{title or ''} {text or ''}".strip()
        for title, text in zip(corpus_ds["title"], corpus_ds["text"])
    ]

    all_queries = {
        str(qid): text
        for qid, text in zip(queries_ds["_id"], queries_ds["text"])
    }

    qrels = defaultdict(dict)
    for row in qrels_dev_ds:
        qrels[str(row["query-id"])][str(row["corpus-id"])] = int(row["score"])
    qrels = dict(qrels)

    # Development queries = queries that have dev judgments
    query_ids = list(qrels)
    query_texts = [all_queries[qid] for qid in query_ids]

    print("Documents:", len(doc_ids))
    print("Dev queries:", len(query_ids))
    print("\nExample document:", doc_texts[0])
    print("Example query:", query_texts[0])

### Indexing and Evaluation

In [ ]:
if "quora" in DATASETS:
    quora_results_df, quora_per_query_df = run_benchmark(
        "quora",
        corpus_texts=lambda: doc_texts,
        n_docs=len(doc_texts),
        doc_ids=doc_ids,
        query_ids=query_ids,
        query_texts=query_texts,
        qrels=qrels,
        evaluate_fn=evaluate_binary,
    )

    all_dataset_results["quora"] = quora_results_df

    # Free the corpus before the next dataset
    del corpus_ds, queries_ds, qrels_dev_ds, doc_ids, doc_texts, all_queries
    gc.collect()

### Results

In [ ]:
if "quora" in DATASETS:
    plot_results(quora_results_df, "BEIR Quora — Tokenization Comparison")
    print_improvement(quora_results_df)

---
## Dataset 2: BEIR HotpotQA

5,233,329 Wikipedia paragraphs, 5,447 dev queries, two supporting paragraphs per query.
Document text = `title + " " + text`. The corpus is streamed from the HF dataset during
tokenization to keep memory down.

> ⚠️ This is the slow one: about 2–3 hours for all six analyzers (index build ≈ 5–9 min
> and Porter tokenization ≈ 17 min per analyzer on the original run).

### Load Data

In [ ]:
if "hotpotqa" in DATASETS:
    HOTPOTQA_HF = "Hyukkyu/beir-hotpotqa"

    corpus_ds = load_dataset(HOTPOTQA_HF, "corpus", split="test")
    queries_dev_ds = load_dataset(HOTPOTQA_HF, "queries", split="dev")
    qrels_dev_ds = load_dataset(HOTPOTQA_HF, "qrels", split="dev")

    def find_col(dataset, candidates):
        for col in candidates:
            if col in dataset.column_names:
                return col
        raise ValueError(f"Column not found: {candidates}")

    doc_id_col = find_col(corpus_ds, ["_id", "id", "doc_id"])
    query_id_col = find_col(queries_dev_ds, ["_id", "id", "query_id"])
    qrel_query_col = find_col(qrels_dev_ds, ["query-id", "query_id"])
    qrel_doc_col = find_col(qrels_dev_ds, ["corpus-id", "doc_id", "corpus_id"])
    qrel_score_col = find_col(qrels_dev_ds, ["score", "relevance"])

    # Preserve dataset ordering
    doc_ids = [str(x) for x in corpus_ds[doc_id_col]]

    query_ids = [str(x) for x in queries_dev_ds[query_id_col]]
    query_texts = [str(x or "") for x in queries_dev_ds["text"]]

    qrels = defaultdict(dict)
    for row in qrels_dev_ds:
        qrels[str(row[qrel_query_col])][str(row[qrel_doc_col])] = int(row[qrel_score_col])
    qrels = dict(qrels)

    assert len(doc_ids) == len(set(doc_ids))
    assert all(qid in qrels for qid in query_ids)

    print("Documents:", len(doc_ids))
    print("Dev queries:", len(query_ids))
    print("Qrels:", sum(map(len, qrels.values())))

### Indexing and Evaluation

In [ ]:
if "hotpotqa" in DATASETS:
    hotpotqa_results_df, hotpotqa_per_query_df = run_benchmark(
        "hotpotqa",
        corpus_texts=lambda: (
            f"{row['title'] or ''} {row['text'] or ''}"
            for row in corpus_ds
        ),
        n_docs=len(corpus_ds),
        doc_ids=doc_ids,
        query_ids=query_ids,
        query_texts=query_texts,
        qrels=qrels,
        evaluate_fn=partial(evaluate_binary, both_at_10=True),
    )

    all_dataset_results["hotpotqa"] = hotpotqa_results_df

    # Free the corpus before the next dataset
    del corpus_ds, queries_dev_ds, qrels_dev_ds, doc_ids
    gc.collect()

### Results

In [ ]:
if "hotpotqa" in DATASETS:
    plot_results(hotpotqa_results_df, "BEIR HotpotQA — Tokenization Comparison")
    print_improvement(hotpotqa_results_df)

---
## Dataset 3: WANDS (Wayfair)

42,994 products, 480 queries, graded labels (Exact=2, Partial=1, Irrelevant=0).
Document text = `product_name + category hierarchy + product_description`.
All queries are evaluated (nothing is tuned). Data is read from `../datasets/wands/`.

### Load Data

In [ ]:
if "wands" in DATASETS:
    query_df = pd.read_csv("../datasets/wands/query.csv", sep="\t")
    product_df = pd.read_csv("../datasets/wands/product.csv", sep="\t")
    label_df = pd.read_csv("../datasets/wands/label.csv", sep="\t")

    text_fields = ["product_name", "category hierarchy", "product_description"]
    product_df[text_fields] = product_df[text_fields].fillna("")

    # Preserve document order
    doc_ids = product_df["product_id"].astype(str).tolist()

    doc_texts = (
        product_df[text_fields]
        .astype(str)
        .agg(" ".join, axis=1)
        .tolist()
    )

    query_ids = query_df["query_id"].astype(str).tolist()
    query_texts = query_df["query"].astype(str).tolist()

    relevance_map = {"Exact": 2, "Partial": 1, "Irrelevant": 0}

    labels = label_df.copy()
    labels["query_id"] = labels["query_id"].astype(str)
    labels["product_id"] = labels["product_id"].astype(str)
    labels["relevance"] = labels["label"].map(relevance_map)

    assert labels["relevance"].notna().all()

    # Resolve duplicate (query, product) judgments by keeping the maximum relevance
    labels = (
        labels.groupby(["query_id", "product_id"], as_index=False)["relevance"].max()
    )

    qrels = defaultdict(dict)
    for row in labels.itertuples(index=False):
        qrels[row.query_id][row.product_id] = int(row.relevance)
    qrels = dict(qrels)

    print("Products:", len(doc_texts))
    print("Queries:", len(query_texts))
    print("Labels:", len(label_df))
    print("Queries with labels:", len(qrels))
    print("\nExample document:", doc_texts[0][:300])
    print("Example query:", query_texts[0])

### Indexing and Evaluation

In [ ]:
if "wands" in DATASETS:
    wands_results_df, wands_per_query_df = run_benchmark(
        "wands",
        corpus_texts=lambda: doc_texts,
        n_docs=len(doc_texts),
        doc_ids=doc_ids,
        query_ids=query_ids,
        query_texts=query_texts,
        qrels=qrels,
        evaluate_fn=evaluate_graded,
    )

    all_dataset_results["wands"] = wands_results_df

    del query_df, product_df, label_df, labels, doc_ids, doc_texts
    gc.collect()

### Results

In [ ]:
if "wands" in DATASETS:
    plot_results(wands_results_df, "WANDS — Tokenization Comparison")
    print_improvement(wands_results_df)

---
## Cross-Dataset Summary

Metrics side by side for every dataset that ran, plus each analyzer's average
nDCG@10 rank (1 = best). Saved to `tokenization_summary.csv`.

In [ ]:
if all_dataset_results:
    summary_df = pd.concat(
        {
            dataset: df.set_index("Tokenizer")
            for dataset, df in all_dataset_results.items()
        },
        names=["Dataset"]
    )

    summary_df.reset_index().to_csv("tokenization_summary.csv", index=False)

    datasets_run = list(all_dataset_results)

    ndcg_pivot = summary_df["nDCG@10"].unstack("Dataset")[datasets_run]
    ndcg_pivot["AvgRank"] = ndcg_pivot.rank(ascending=False).mean(axis=1)

    print("nDCG@10")
    display(ndcg_pivot.sort_values("AvgRank").round(4))

    recall_pivot = summary_df["Recall@100"].unstack("Dataset")[datasets_run]

    print("\nRecall@100")
    display(recall_pivot.loc[ndcg_pivot.sort_values("AvgRank").index].round(4))
else:
    print("No datasets were run.")